## 1. Dataset Overview & Data Quality

In [1]:
import pandas as pd

dictionary = pd.read_csv('../data/goodreads_data_dictionary.csv')
dictionary  # just display it — this is your field reference

,Table,Field,Description
0,works,work_id,Unique identifier for the book (work) in the d...
1,works,isbn,The 10-digit International Standard Book Numbe...
2,works,isbn13,The 13-digit International Standard Book Numbe...
3,works,original_title,"The original title of the book, in its origina..."
4,works,author,Name of the book’s primary author.
5,works,original_publication_year,Year the work was first published.
6,works,num_pages,Total number of pages in the work’s primary ed...
7,works,description,"Summary or synopsis of the book, typically fro..."
8,works,genres,Comma-separated list of genres associated with...
9,works,image_url,URL link to the book’s cover image.


**Observation:** the dictionary confirms `works` is one row per book with pre-aggregated rating counts (`5_star_ratings`...`1_star_ratings`, `ratings_count`, `avg_rating`), while `reviews` is one row per individual review. `work_id` is the shared key between the two tables.

In [2]:
works = pd.read_csv('../data/goodreads_works.csv')

print(works.shape)
works.head()

(13525, 20)


,work_id,isbn,isbn13,original_title,author,original_publication_year,num_pages,description,genres,image_url,reviews_count,text_reviews_count,5_star_ratings,4_star_ratings,3_star_ratings,2_star_ratings,1_star_ratings,ratings_count,avg_rating,similar_books
0,2919130,1416534601,9.781417e+12,Nocturnes,John Connolly,2004.0,NaN,NaN,"fiction, fantasy, paranormal, mystery, thrille...",https://s.gr-assets.com/assets/nophoto/book/11...,8820,338,1118,1601,1029,190,58,3996,3.9,NaN
1,52087333,NaN,NaN,Draw Play,Tia Lewis,2016.0,NaN,Jake:\nI can't believe my coach assigned me a ...,"romance, fiction",https://s.gr-assets.com/assets/nophoto/book/11...,2482,204,204,353,274,77,29,937,3.7,NaN
2,1649583,1416505520,9.781417e+12,Citizen of the Galaxy,Robert A. Heinlein,1957.0,NaN,"In a distant galaxy, the atrocity of slavery w...","fiction, young-adult, fantasy, paranormal, chi...",https://s.gr-assets.com/assets/nophoto/book/11...,16506,447,3539,4351,2863,444,53,11250,4.0,NaN
3,688299,0060541830,9.780061e+12,Congo,Michael Crichton,1980.0,NaN,"Deep in the African rain forest, near the lege...","fiction, mystery, thriller, crime, fantasy, pa...",https://s.gr-assets.com/assets/nophoto/book/11...,170916,1633,25081,45775,48505,14001,2926,136288,3.6,NaN
4,3464264,0451528824,9.780452e+12,Anne of Green Gables,L.M. Montgomery,1908.0,NaN,"Everyone's favorite redhead, the spunky Anne S...","fiction, young-adult, children, history, histo...",https://s.gr-assets.com/assets/nophoto/book/11...,743392,14586,272952,161856,81578,19933,9099,545418,4.2,NaN


**Observation:** 13,525 books, 20 columns. `original_publication_year` loads as `float64` (e.g. `2004.0`) rather than an int — will need casting before any year-based grouping or plotting.

In [3]:
works.dtypes

work_id                        int64
isbn                             str
isbn13                       float64
original_title                   str
author                           str
original_publication_year    float64
num_pages                    float64
description                      str
genres                           str
image_url                        str
reviews_count                  int64
text_reviews_count             int64
5_star_ratings                 int64
4_star_ratings                 int64
3_star_ratings                 int64
2_star_ratings                 int64
1_star_ratings                 int64
ratings_count                  int64
avg_rating                   float64
similar_books                    str
dtype: object

In [4]:
works.isnull().sum()

work_id                         0
isbn                         2051
isbn13                       1661
original_title                  0
author                          0
original_publication_year      18
num_pages                     730
description                   169
genres                          0
image_url                       0
reviews_count                   0
text_reviews_count              0
5_star_ratings                  0
4_star_ratings                  0
3_star_ratings                  0
2_star_ratings                  0
1_star_ratings                  0
ratings_count                   0
avg_rating                      0
similar_books                2515
dtype: int64

**Observation:** the columns that matter most for the five planned charts — `genres`, `author`, `avg_rating`, `ratings_count`, `reviews_count`, the per-star counts — have **zero** missing values. Gaps are concentrated in fields we don't currently plan to chart (`isbn`/`isbn13`, `num_pages`, `description`, `similar_books`), so missingness isn't a blocker for the chart plan.

In [5]:
works.duplicated().sum()

np.int64(0)

**Observation:** no duplicate rows in `works`.

In [6]:
reviews = pd.read_csv('../data/goodreads_reviews.csv')

print(reviews.shape)
reviews.head()
reviews.dtypes
reviews.isnull().sum()

/tmp/ipykernel_23703/1195728872.py:1: DtypeWarning: Columns (3: started_at) have mixed types. Specify dtype option on import or set low_memory=False.
  reviews = pd.read_csv('../data/goodreads_reviews.csv')


(1143887, 10)


review_id           0
user_id             0
work_id             0
started_at     347495
read_at        112122
date_added          0
rating          37318
review_text         0
n_votes             0
n_comments          0
dtype: int64

**Observation:** 1,143,887 individual reviews. `rating` is missing on 37,318 rows (~3.3%) — these are text-only reviews with no star given, so any per-review rating analysis needs to drop or explicitly account for them. `started_at` triggered a mixed-dtype warning on load and has the most missing values (347,495) — it's the least reliable date field here. Given the file's size (881MB), full loads are workable but slow; may sample for iterative chart development.

In [7]:
works.columns
reviews.columns

Index(['review_id', 'user_id', 'work_id', 'started_at', 'read_at',
       'date_added', 'rating', 'review_text', 'n_votes', 'n_comments'],
      dtype='str')

In [8]:
works['work_id'].isin(reviews['work_id']).mean()  # adjust column name

np.float64(1.0)

**Observation:** every book in `works` has at least one matching review in `reviews` (join coverage = 1.0). Clean key, no orphaned books to worry about when merging the two tables.

## 2. Resolving open questions

Before locking metric definitions in Branch 2, four things need answers from the data itself rather than assumption.

### Q1 — is `author` single or multi-author per book?

In [9]:
multi_author = works[works['author'].str.contains(',', na=False)]
print(f"{len(multi_author)} / {len(works)} rows have a comma in author ({100*len(multi_author)/len(works):.2f}%)")
multi_author['author'].tolist()

1 / 13525 rows have a comma in author (0.01%)


['Garret Weyr, also Freymann-Weyr']

**Decision (Q1):** only 1 row out of 13,525 has a comma in `author` (`"Garret Weyr, also Freymann-Weyr"`, a pen-name annotation, not a co-author list). `author` is effectively single-author per book. **Chart 2 (author-level aggregation) can group directly on `author` with no name-splitting needed.**

### Q2 — is `genres` single or multi-genre per book?

In [10]:
genre_counts = works['genres'].str.split(',').apply(len)
print(genre_counts.describe())
print()
print("Pct of books with only 1 genre:", round(100 * (genre_counts == 1).mean(), 2))
print("Sample genres field:", works['genres'].iloc[0])

count    13525.000000
mean         5.869945
std          2.457028
min          1.000000
25%          4.000000
50%          6.000000
75%          8.000000
max         23.000000
Name: genres, dtype: float64

Pct of books with only 1 genre: 1.33
Sample genres field: fiction, fantasy, paranormal, mystery, thriller, crime, young-adult


**Decision (Q2):** books have a median of 6 genre tags (range 1–23); only 1.3% are single-genre. Taking just the first listed genre as "primary" would discard most of the signal and the list order isn't documented as meaningful. **Chart 4 will explode `genres` into one row per (book, genre) pair** before aggregating median reviews per genre — meaning a single book contributes to multiple genre buckets. This will be called out explicitly in the chart's write-up so the genre totals aren't read as mutually exclusive.

### Q3 — should Chart 3 (controversy) use the `works` star-count buckets or raw `reviews.rating`?

In [11]:
import numpy as np

star_cols = ['1_star_ratings', '2_star_ratings', '3_star_ratings', '4_star_ratings', '5_star_ratings']
stars = np.array([1, 2, 3, 4, 5])

def weighted_std(row):
    counts = row[star_cols].values.astype(float)
    weights = counts / counts.sum()
    mean = (weights * stars).sum()
    return np.sqrt((weights * (stars - mean) ** 2).sum())

sample_std = works.head(5).apply(weighted_std, axis=1)
sample_std

0    0.919003
1    1.003609
2    0.876496
3    0.974137
4    0.945770
dtype: float64

**Decision (Q3): use the `works` star-count buckets, not raw `reviews.rating`.** Reasoning:
- `works` has the per-star counts for all 13,525 books with zero missing values, so a weighted std is computable for every book with no gaps.
- `reviews.rating` is missing on 3.3% of rows, and Chart 3 needs one variability score *per book* — using raw reviews would mean grouping 1.1M rows by `work_id` and handling the missing ratings within each group, for no real precision gain at this grain.
- The weighted-std approach above (binned across 1–5 stars) is a standard way to approximate rating spread from aggregate counts and is demonstrated on the first 5 books above.

**Chart 3 will plot `avg_rating` vs. this weighted std, computed from the star-count columns.**

### Q4 — what minimum-ratings threshold excludes low-sample noise for Chart 5 ("highly-rated authors" over time)?

In [12]:
print("ratings_count percentiles (book level):")
for p in [5, 10, 25, 50, 75, 90]:
    print(f"  p{p}: {works['ratings_count'].quantile(p/100):.0f}")

books_per_author = works.groupby('author').size()
print()
print("Books per author:")
print(books_per_author.describe())
print("Authors with only 1 book:", (books_per_author == 1).sum(), "/", len(books_per_author))

qualifying_books = works[works['ratings_count'] >= 1000]
qualifying_authors = qualifying_books.groupby('author').size()
print()
print("Authors with >=3 books at ratings_count >= 1000:", (qualifying_authors >= 3).sum())

ratings_count percentiles (book level):
  p5: 663
  p10: 1112
  p25: 2673
  p50: 7351
  p75: 22025
  p90: 65326

Books per author:
count    5554.000000
mean        2.435182
std         3.432484
min         1.000000
25%         1.000000
50%         1.000000
75%         2.000000
max        53.000000
dtype: float64
Authors with only 1 book: 3340 / 5554

Authors with >=3 books at ratings_count >= 1000: 1228


**Decision (Q4): threshold = `ratings_count >= 1000` per book, plus `>= 3` qualifying books per author.**
- `ratings_count >= 1000` sits just above the p10 mark (1,112) — it drops the noisiest ~10% of long-tail books while keeping 12,349 / 13,525 books (91%) in play.
- Books-per-author is heavily skewed (median 1, 75th percentile 2), so most authors don't have enough books to plot a trend line at all. Requiring at least 3 qualifying books per author leaves 1,228 authors — enough to show genuine over-time patterns for Chart 5 without single-book noise driving the line.

## Branch 1 wrap-up

Both datasets are understood: `works` (13,525 books, book-level aggregates) and `reviews` (1.14M individual reviews), joined cleanly on `work_id`. All four open questions from the plan are now resolved with data-backed decisions:
1. `author` — single author per book, no splitting needed
2. `genres` — multi-genre (median 6/book); Chart 4 will explode to one row per (book, genre)
3. Chart 3 controversy metric — weighted std from `works` star-count buckets, not raw `reviews.rating`
4. Chart 5 threshold — `ratings_count >= 1000` and `>= 3` qualifying books per author

Ready to move to Branch 2 (`eda/analysis-definitions`) to lock these into concrete metric definitions and summary DataFrames.